# Đề 12 – Kaggle T4 v4: hybrid DINOv3 + SAM 3 (sam3.pt, fp32, class-vocabulary prompts, masked-crop region vectors) + EmbeddingGemma 2 baseline; 70 + 10 video. Built with DINOv3.
Trọng số offline từ dataset private `holilihu/de7-weights`. Toàn bộ kết quả → 1 file `/kaggle/working/de12_results.json`; dữ liệu làm việc ở /tmp.

In [ ]:
import os, sys, io, base64, tarfile, json, time, subprocess, traceback, glob
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', DE12_FP16='1', DE12_SAM_FP16='0')
OUT='/kaggle/working/de12_results.json'; W0='/tmp/de12'; os.makedirs(W0, exist_ok=True); os.chdir(W0)
R={'started':time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()), 'precision':'DINOv3 fp16 autocast; SAM 3 fp32', 'stages':{}, 'errors':[]}
def save():
    json.dump(R, open(OUT+'.tmp','w'), indent=1); os.replace(OUT+'.tmp', OUT)
def run(name, cmd, timeout=3000):
    t=time.time()
    try:
        p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=W0)
        R['stages'][name]={'rc':p.returncode,'sec':round(time.time()-t,1),'tail':(p.stdout+p.stderr)[-3000:]}
        if p.returncode: R['errors'].append(name)
    except subprocess.TimeoutExpired as e:
        R['stages'][name]={'rc':'timeout','sec':round(time.time()-t,1)}; R['errors'].append(name+':timeout')
    save()
CODE = 'H4sIALx/32oC/+1963rbRrJgfvMp+iA5x4BEggRvkqkwG9/ieCeJFclJZj6GBwOSIAmLBCAApKgja799mn2wfZKtSzduJC3PjOM5O6byxSSBvlZXV1dV18Wsm/Vvz53N964zcaMv/pC/Bv/t+2w0Wq3sOz63Gk3L+kJsvvgEf6s4cSLo/ovP8695IpaJt3T71snjVqNlNa1T0+o8brbblS8Of//6f+vYdaLxvP5H9oGb+qTTwU/rpNPIf6Z73uo0uyfwX6PbxP1/0mp/ITqfcv+PgveTm4fe/3/6Zx7o/4H+S/rfPOmcdq2O2Tlpn55YB/r/OdF/2/Z8L7FtM7z9Q/Z/t91+kP53ux2r08L932o0W1+Ixqfc/wf6/0+k/+1t+t880P9PQv9Pi/x/G+j/42bntH0QAD4r+j8OIvePoP0P0/9mu3HSTPn/1gnu/2az3TnQ/0/xp2nar97EDYS3dGZuPRi9dceJYKToiWnkLN1Y1L4R+nS1WNTotzgWkTvzAt8Q7nLkTiaeP6My3z15dXmJX2J3tnT9JDah9Yq3DIMoEW/jwK+KIK6K2SIYVQUSHfVuvG5Whb9ahrfCiYUfVsXU8WIomgQwjMo0Cpbi/NUPQhZ/hUPlp0nk+PE0iJZuFKvXT1ZJ8GMwcRdV+noeBWM3joOIf1Ll9Fml8uur5y9e2y/+/Eb0ha6Zy7CtVYVmOmuPPpfBmj+v+PPGHS01o1L5UtRKfymwSn+ViTsVztyJ5/o0MnoVAX8z6AwmbUZu7P2Xq+PX8Tp5FiyCCApV6d2z1z+8vrCfvrxovrx48hejKvTTqjiFT89P3CgMFk4Ca9DHoq9+evPiwn5y8eKJQc1HbrKKfKHPxDdiZi5dx9cNw5xCjcSFrxUaUuwsw4Vr86j10EnmAPYw7ltmoyom7mQV9htyuLCMv/gegplrwYKbXOSbRk/EV14oHJ6+uJkHsSu67drIS4TzPcxaeLG48ZK554u/Up2/CngXi2AqkrkrFk6ciCs3TLgB6k8HjHTEKIii4MadCFrp597sl9AwxV88dzGJhZ7YsTuuIlqYtKQGIRvWphb74qfA59bGTijBTZj+zAkBOi7N2DgTa5gzvnZCc+YmtBTPnpzb5xevz+3vzi8NEUSi2TEb1FScuICcHpRfOhvdqoooWPkTndqoI/QMWB8uejP3Fq54E61chiH+BVcA4kj2FgHHpRvpO28q/CCBIj0xgldX+Ree+HfqWvT7opE1h39zaC1FLixLIBbuAlYhhUCuKX7t+BMGE6wN9ool6SHglq7Pxb/16bVhxqsloI74us8Vi31Tk+K4L6wzMQ78xPNXxf7kSswLD29xAaFenUBf5d1s4ho7UeTcPrQZLl4+NXJQo+7VMgNMF64Tu/ruDcpUa88OnUXexJYldG85q4qFuwYw9nWgTS1Y1mDtRgsHNoXZ7GT74rmbAMEMIiCNrit7EGEUhEHsLOKeqhXClhHL1SLxavHYAcTA7kxxQRs1FgsPAAUbQt80qreN6saq3loZQv9WFd8DGGFQJpKLM9gZGxeRdjCk97AxhQ9LJ0ecLdL4BqCHGPIbQNvHVuDjTITwOMTH4xtxpEZIJdNfaRPYdoRtA6GdubpvFFEAX4/3v8Y/GqwJIHBhp+i4bxpEw/Tx0fimFt7glsk9jY7G81o4p6eer//GT/XxsWVA+WNZHt58L99E9GZ+jHWMAv2jniuVMaBhLF7/9sO6ecHrm67e6xBpqLMQAYyutg7GzkgegT4cD9k6Cv3l+S/Q7DhYwsEGWwEI0SwIZgs4MW8W62ZtBFhXA4oynlvdmuvHcDAu3HQBEb+UgK3H7mIKpx3Qur72cBsaHoLhVf8UafLaG0OlcbjScnDeew6+xlZzpx/9/i6IXtMEGXFh9mlDODATSWWxIm1NOwSYRo7nuxMdhw6Ek4ovVfHtdnfXM5NA54kYprt2FjkKSA0ChaRPKANN49TVxKnct8QRmH5gz6KUfDJ4YVstUvDCXsmBaANN8ex0YnPiPm1wRhMbzkSYZtzXwkSj8akBZEObLukM4Smb1IbNvI8b6aG3cRc2zGUFDW8GWv63NjQGjWHaDvA9cMrC9ptAa9ioGc+dEHb01HWSWD0DrgCf6lj6CAtnAwHkzMaRYSqCeeIhHdKpJSDc3mwZeAChQveSbsj68LNcsUojwEpCfCnGm/HtzRy3d+jAVCe1+HrlRC4Q+yCCM3jQqFpZ45fyVFRkClAkylMpRS7eYnswdBMXl2F9ZZieD8MAOpEESAv1Eh0Zb4A83VbFCKAxQsql01QGb4dAsi6NtFahEtBSAcRUADUVt5YcHfweb2qjm3ozIzvj29pozg+ArqjxAwyw7LEqm39l4avbY6pVPmE3Vm3TAMbL6tKJemvVbvlnD+GREkKkXZsGs3P6rfqysdQTy8gdc5KgRUjOdhxsChW3TzamfZcesnrPPT9jhlMS+AY31NqLYcfyGeZerzxAXuDf8UR6/uqn1+vWr96bIt+Me86B4yymSQFIBIhsG/gfeDtgD0F8kAf7T24ikAGtx8nkffQQ2+hD/dzCE2pQ233B5x7jLbQGT/zQZH5h0DDbpx1gu8x2p0sfje5Q0SfotVS22XyMhaAr/ugM30NEkFbsphNF4gIdgBg5vtIHOvYVc2/e0gS2CA7URNeAbQHiIvl9PZ1aNZuloVihp69+ePXTiycXwHfVgfUE3lPUsqnjw3RquJ+8JW4oHutwC2XuiuSoxyKVyXPRNyYdHHDIubgRWlUBewWxfZLchm6fy04XgZO0msa9OkpfSHRLcWi88MIenaGOV8cftbWX5M6zVlO8gyWcwZtmTx2c8nfx2EMMeidAogzWUHLqjN1REFzV+UEtXsLysIRAT1rCNUESKRZrYecxtKWOntpiPbG6p4+XyO3OnMSd9ITvukDAvv8OwHHl+iDRLoD6wJErnPEYZBHHH+cEir3H9/tmrKUHNjLXZbTG+og38HGWP/C4CsocujZeTRwNKQovA/40vdh21o63cIA10A1m85khyMsLWiqXz9zlEhvxqStzAfJUpJNMo7mzJr0IYhNFIRMHj4X0OIn4rDZU+R4eBi9Umy+xTdEUOvGzy2DioKQdOuO5W2uaDUOsPQfmBFIqQLGW50yKrCPyLaqYvYuBuZQv32TvCi3w2QEnBwBOixMtx5PsqEpzShdFwRwIO6oK7KsbJ5rF/TuNMD/dJxL37+kwww1VXEbsKlU37Gd4UoZiN8sjZ4DYS+utq687Vi19A8IqiPORLs9yIDRTD3gajadDk4BfmqGQZL3Q8iiS61l1XTxyk+i2l7GE25qT3bMtNOFucC8BVcPVfAHCfJRrcceZBFMEPOMTB3lvedTwXk2ycwpO/4i2bUA8ftYpzrQ06A8Z74czlXwejOI+kMPefnACLhaBKYlx7myQywY4CuulD+40ahzwbv+hQUcWHpHG/Tbhh1Eh3bHpHB3BIH1AemcBv+xMSddHdURK3TWJ3DnaEaySbYbNy+S7BkrFvmSiDYREiVHbxWnjx8DreccjHOauw/SM6t1d9cS6sF2oexAA1jiCjekl7jLW4RwqcVwP4rIgBZTc0g5gxdgBbpEJ7I6zzgI2AkghkNhJX8+IM7TNJBnZOqCarr/2IpBzUG2kPX9hNe3vzq0u7jpLM6g4fO4QiOcpB/6Q6GCiAsWee3DY+jYwGIm71ZgfuahM3EcN/NWSlBpx4kY2nXUxDLBBtKSR221zJxZtoUryqRhvdTYlkYwOIyfRB/NBD9qCNaUvFpyiOJrekDWOlgFvahbt6Wc/XMJb4tzoeCySCdy1O/uSs4HJ2SxzoYyyilBd+X7AbaupWLsG5MOP6XTXp1K7bL4hTIQzDnsExA2CBcAKdkK4SgrtwCPFvUuCATLuyh+zCsFMN5w+ZTTScad5yz7AwIQjWjdMUnHr23w90AVYMYApYF3i6tCRAYQC0VLPbdJdnD8gvbvZrdAarbzFxKYC+hoVn/bEA64cGucvdFA8yMOkGmGpGutrqLmCF4tg1g8jEFXKymLYGUvnCqTKKNbTztwNIJYdXDEFOkOJBaCtGEmdxsJgiZ1lXnmL3Aw8atGZF+XVNykPIa+wTCymNHgpC/Hkx5ZU+qC4F8O47HEUhGeyn9x7HYaFCh/cTTldII5AfcU9je+OeUTbStYLGxpEUqaRfArkvMHUqojjspu8SkoqRfoAGCZ9pW5JRSS71B8aX25oDNS1O8b5A5GAgwExv0Fkvkr/N/gyxsR/5OGHdJeILuwM4Jf1KT2ZEt+xCEYm/qMrxvFtAKJxDsO0I+Q6YFhTxbkAqZzESIL19LYlTxix7YR0NtRl4V5iTSgocaysFQAw4rRwLlAZp3On0UAA8Fts7Rq2owYHs9TZJ6zV1fDcgIca9YfEG1YOfiPs7oflkwYXbgepgvGPSEsCr/Wi6mlrvIqEoC4BH+gjAxBvqR5nE1gOGsOB/DWUQ+dnCf1WA+fFz0Y+ut9SSQBI94waoHeFI/8vL9ShFA2+iuX56MEdE3/IbHJbi5Xn1PJHmZkNo8lmR3qekYEMOW22gdxpw+wiQP0h0qvOYVvpNGAaEuwDLL48g82Q1fszC/J5SoxNQIUJckW5PQLyeNJINwpxHgBDPcOhbE1yc8hzjJO4oJbwmNGqAh8EDblwTrgR9o8jpa0ERxxBZYh7HL8NUy4OSHC3XZT/iEebxAbfF23fzXgT5LfojtV8hQfEq+c/OqGee/Ddwklenet/Zv3kwBoi3KCa6UwmNu5kG9rX/zyAf2Ek2Fe2m6mVmwi4NXn6QD3YxdqdPA3u63c4g3uTSsqRI3cTE+WkswAxgq9wNdj+8IuIgIQm/o4UUdeIMsAT+qzKjYxFkNBpa7qdwZ8IFIInPKVx4QViSg4mQA9Q9yar2yHwAPn3/rSORZqS+5SbiodtrsIJrhY8sQkb+0XUpA3Fb6hd6qLP7RZL1uW1oj81WA+DFYEBmPRpS8IXAOk04OdAbZZhEvdpS+B7+UDyF3jhbk6A49Dvjo5onCgdAgAQtvBxXyWNjZ5fGHxuYkVE3RtNtgRHvU4NpFIwz3snR8KH8V5NJL1VqqMdihVeGeYZJHewU4GCEjPNEIGiq4mktfNTMdLLCtz3kqucDBgWw2KzzJek+lTYlCmPklYj/ByWhHjeUijCMP7j7a5E/+K47q5SxGfq8UGkAxFOnWjES8X7m72vpMDlxcjuRJjr66e9XfWtBl0w+f1mo4EcQgRc9swJCeZVcQ3bpwx+TSMQSOMQ8U5der4DWRxGl95pgsQIR0JqCwJnM1H8KtrbJFUgytXQda7spBqj7VEVdsD9MFW54d81ABQHgHPnz9xlNTE4asl0Ov5hmUcuXTjfZSIiUhqmKzznnhjIb8MM2PhQfh3SSZPMcwWzcvgSlyMrT6szrG4fkFz3WJUutSSrQalV7O7sqlT0foAwL8rl6twZXGWIxNMFgNEvhZnD4gn+vCpeqX0Abwd0npgSV675qgNxolou4idB4iwMY+sKGEjLW8VGPKeLk1fw73tlX9h8g7fDM3Hl3uJ9zjzHEMyJEdgpxGFpEuR8XmvsW3xD3wfwbgi99rJf2DCLYqismBOZzTc8o1u9FOmxg9yPMrZZZkfp3oEM4LE0LAtyXFuhvI4DqWIvxqB3NWSzm71FUm36j1hCJC5KMM5icSucyVtnjPcxc7SbQVLvCt5L4t0kefd1H6vjrVGQ7rYz9U3Q7uLLL6zPv9N9Nrq119mGwaVEjjtBayNUdBkK0Er30qMagCuJvADC4oOhkV5rwWSoojzV3VnJTgGKz2NqFdtRraZwnMcmihwoi41XUV4QzAkK2AE2MY97ZU4XK6F6JgEWDb4PNJeYpq/7COJtdMwVAQZvF8IhdlEpgpsG2AW/0gMfn/VhOEzJ+jA2wLD+yPgA/QYPtkcgUtCDB8auMRL9UtzzGokQ0lBUUsN3l5hl/MajkD94vD0EVioY5I6sHZ2qo51FPixSxf3WXzjL0cQRm56obVI4GAfr4IP9/8H+/1/N/r/ZeNw2T9qPG+3GwQHsc7L/j1zk1/8YD4AH/L9ajc6Jsv9HBwC0/+82ugf7/09k/88aaDj+a4gCaKiKptlBWPtTyk72iPVyHRDsx8Beecj9qJd4NVBDAVwgd4o8CEuGVbyUEtKhILWkrOioIW/h1e3L81/OSNWOP56d/4K2/3g5M6l5viHFf9kn6sv5Cm/uCuJ/+eI0qhK3F7nM5dJbyfc6cQWVKUHkzTwfLRTgF+kn7XEQeyDR5iRVt8ZMTyasHqO6D83ccx4M6LuApLK65a+wx0OB+H1SBBBApI6cLe0TyfTuMU7PiiInjHaCZdv089eX9o+XL55BU+JIWLCPoCQZmBfsy89KdtE5Ru/9htc77a5JDXGVu/io2M+ePPv+BYkQeNGnS7mEWVEDnUHSpT/jxQSWkjUHnjKwdREDxi5UitS1v4IdYqQeS6VRVeRuGUibIetHSq9BkjtwrbaHCg38EqwS0nGUNRl0A5JT6Z59kLqBx5FTOeSlnPS1lKRZhYID4xHJQVxDXz4q78jvIFMjxyyxzwrXW1Aih0D7blziVHg26IcUBIYFpTA0BjPCQe3QCEspPM5J4ayfzLWGFyaFFudOTJfO2Spo0I4GQqJ6YnK78G9ltxDP6NOTn0poV9XRp2HLeDVXNAPkcZ/0u1Rmy6eCnu6Y9Iv8otGaTiN1J5Nd6lBtMij0ltj/C/GtuB40oHM0xUVbTXxhOtEM6U0RQvgGjVS/wVVJJci4KD9KDQUXNViE5GsOaeNqVOVOcCd8f5piHQvL7xHS8nIdlB/0aEfAUt5p3KRUcpcvN1CaVPsWVekM5vsDS3iQ/w7y37+8/3fj8Wm73T1s9s9I/ssbz3xsKfD98p9lWd1Gyf8b/m0c5L9PJP+xQFZ2XSSXtzOSy26C6Ip9eMVoim4t0oISRKh1W0w95K4qNfHS9d3IGwu+hhYai34a3zjE4n81zEazBv+cgpQABace8f3I7nAv2H4wnXpjz1mIJFqNr8y34awnNPquiYZ52m1Vs3ahqZPHBt69xYG4ccUKzfeRyyOrY3LmWy2c6JbcWJ78+uqy/uz1s9e1hXflCr6N9slhW3/+4rsnv/zwBuWqH8/fXOKlR5zAeWjCpDKPF7SnS3iUZF45Ybv1OXprBYtJH4YjdBSB5c0IjK8DDCA60ixuxYTYVOZPz3C0U2gN5B1qDyRNmyXYYBXF7mLtxty3OyaDZzR/aDXF6DZtWyd7U1g5sjntWyCv0/ynYX59zpRkjYJx5IaBtKUurKJZSc2c0foHRTbljrq53dzymOUNemYUJHDOMTW8RGNBVBuMgmAh+K3+/eY3lpCZk2b7IGkzVJKpkS93N8nCG32QUF1aLXLbB8lRGi04vrd0FvhtEszwYx4ARPHLOLghMyIvmtBPJ+Kf49sxuVlqywDNOtWvETDl9Amg0ciQZO7hK0N5weRMB/daMygTDRYFlSNnutyAM62Cmwg64brjqxDku4Sk//LluwQFx0UoelIAATfZ+J8sP91IFWZDUCLwbEVLpfbVNnMFMxt8ZU4J77LYCUWDjI/lv1KCAGozSk+g9bL5NS6G/exP52+0zPMLaBEaA+wsevnm4tWzN7iujdRWO5M0Q3dfvafnL3JDJYBB0d0Q1vMOu2gwS3J8sLS//67/HVBYWO2jI/1Og+5oXnhFF7r3CDEcAcHm7t4oGZqkRkBFOwe8TiyCqWQreZXabpPFTKkwqsdCewEEgYJKyCHfuN5snsR24C9uSfKEYlBOGfFSm0ClCTxsFgMIfFW8wownZBdjAvlZOGNX1ybSY94k/xS0uuqJdcHSH5qUF8OkLTDptpPtR3O1S34AqCphBCYIkbW8ja6tejypCsYGBrux7URUAKoGgEl2WZCWYGaQ8RkPxwYZXBmZxRN8s/TiGIiifBaZ8jcVxPcr392EUNedpEWyR7LUTrvLnI2JapNNZtFWrdDNoNcZFnrKFyx1hmWLACXjbkZ7wt2eVi0BCy0jAf/H6LM92emUITch6miLVQcpfIYwGMcDVL9Y+SiIkJ+QPuV+ZX3qXsga/btSU4/ki0fD+x17k3HiQScsJHVoD54ncLo0QdxyGdt3/uc8kfy1He4jIvLk0raHgIcF7tMVrJKufiMxpfbMOFx40EiV7cG4DyITZQ5GEUHlTU9klJ3p08OnRFbSk72aeyBZN2Urnh1z42RDAH3oaFKIkDrQAF3pKY3Ulj8OEZ28O85I+uMUlGo7XW8UK1Qk6fu72uv6s+WckbEnpr9aLOTPnJdY5NzkLex42UoGc85GGuIcSz0m0jtULEtmWV+hFhwYPL7gAB7ilum5UTCJ2wljYiEJyrQshnJsgc3hRoSkiMdll3oylEtRH+8X+OwiJhAYx623OGtpcypRs88fVbIGhf2R5BAacV/qIWNtqBxitr2T8VZGmaqwxlmqQ03Wpipz5lgyCw30RSHDfd+e4fZDlzkMAxKLb/oK2VXcmvv3hIfoyfsPDh70ABpvBTL4A2C+ZVQXpiZ8khJsGxF92CrtWqCSFwC5UUgDviA1R1bLlsZXqO5a1OJboiHacI/TAFtUpQtVDImA92tE+XBBpfcfMQQjjL9QcJvap/luoKL+ysVQSWU3xpidEtiPKCaX5ph8UMVPP14KB0QTjvhCNFd/FfwivgFsO0GmFgbloU/AlmMF4JRue8EKQ3VcoZU8WpphpcwgE8YCDA5+5Oc5Kk4oZ7dPFZB8MQEvR4Mqi2EwzatBE+0eiz5mxDnBC3PiwU7A9lq8gehhsWhhsMMdXWXHwBVGBtlTXu7pAQKiXIau8whSDloHMnJ4mywoB9qcOjA0eAv/4iUEvnDIlnSEy4r+B7fl8i1628qVt+iJ9FfA+5mNOIKK1N8KZUXspIbtGkfUQM2hdTsWOnZcG/EbbLRG7UDLXv4aBQNXrRC6K8SPlChJ484tN5hgw35nVRT4F33dQideCz1MrUbbyIw+n+EVNwjxqA0h4kIejTG668XAatC5gIoabEtIXpoODBVlo0belbPIuUUJXjlRg0QObQD7nl7Eo7Duxump4kg/lDRkxazkfmzAb8SSM+EM/hf2rjwTUdo38LYOJ/a+e2XHkJdq6Kp90Lge7n8O8d//G93/FOK/d0+7ja75uH1qdU4PW/Vzuv+Rqp4/xADwAfu/TtvqyvjvJx2L7f9OuieH+59PdP/zejrFkF1K28cK4T85sxmwxhMncUCsAIbYDzBc0iS48VHzEhumuFyFKCqhWFKDdzVgC6DGNEDdM1TgGBTkggfc1ZEZO1NXRv0whOPf3sxBUhIg8bmRqF9Rd3XPp6ALNYGxapGxcPmSZiIefU3KmG9s+2t8/M0joVPspznIHgtvvqpP3JOanAHdYURuzYk5huKEfWNUJ3id5fmz+k1dtlnniEW3SwDDVYzdk6Ilp1CNoX8cxzd26ESJ/dNP0P2xCFT0SPUynjvNTtdMNskjGkPOlXj/GKiqGSYVZY6ISkvmAWWQqzqp5y0b9ZLyh/qOYRuQp5vH7PaCAXWD2czF8K63GLAid9fCcZfxxoVjMWPUWLp0ieerxFugkRFzsRMPGH5gAXVajrivFZYH9cIw+uxpOpdiVAi647tFmLC+mFldXtO+8jIJM0k3CyxAno3Y83396Kh+pOHIxqso9tYuN5X3hvRibFIPpTTFEWzuNJ4+utahf6ssG7kL/NQxhi91YNwLPX07AzzH8B2hUW8eHTUbPdOa3osfnxpaNkga/jDzllpQBK1VMlAdDnvkLqsBUw/QMLYiYiCwyuEwqNCXjPNbeF4pqAOo/0y0HrGKsaimDouxwGybYmeMigL5kg1FR0qnCKVQFX/GkSm1Oxznff1uea+dFcY/2Tn4chDjLU/VCbQ1vUfTQHgndxouQ+6VhMLW1qPnuO3ioqPaLngsyV7SpBtJPdJ089jgHav/Pjk2vtKqu6BVANeyx33lXdxSu0uAgKwDQp05i4JVqFsYjzbnARdmETQQWBRlqirCmKNoYstbTm8T0gtlQMc690ARtMr74Qr1SmoWvkNHZ+gJuhVqNyPNwNvUm93hH8JciI8whsVhSkASH0IWZEZ2rUa/+gjaotillvj6a9FsF6JG4S6jYRNhsiPnhlzqYBSZOmGOsmZhl0/UZCXlPCqGMcAaSNUySlKarePTBR+OEF6i5C6tniVWYxxTuoMI9TSW3Rn5oEraZ3K/pcChZShGEorTPUE0FqTogGWNdFZG9cSUB6JAVRUjDZF/riw/oY5RDi6UwZBHBVuMgKjPzbm7mXgzF5VsqMrBicNEkNCoOrh2WEnAvryjxh6VG8vuR+bTjALfbSP3wzQ5d7bvoM7HxXqM1cVK24pIXPA9sWV2bT+twFBohiGjnoQe3UpKpeDRFYZ63MAW5liZ5C8Ns5eqO4q2MUnj62EJenSFezqLnoPnqFw3PpH72AtH6Vu3yDiAIk5qFFar8I5CQOE3e7Sr0ggrVekUl6/xa1qrma/TVOEYaIKDsEQEKTxNFmhH15CAYMaCMJlDzSGsyqB4zYrTKtyzppW1bBcbQwWCgZZxIYSXDNjcOGBgFIxIGcBHVGNg2sP/Yb2jyu8oZGS4cDe7SbFaii34c+cP9v2eJjkwH5o5YMS5fPNSbQW9fIYi78H+92D/m9n/Nrudx4/N5il8bx0UQJ/DXzwG7iiJ/5vkf+12Gu0Tyv96Yh3yvx70/wf6/4n1/5221TI7p82TzuMD+f+c6D/G9Zt4ICs7FC/vo94DPOT/Dzgn9f/d7km7C/u/3Tnkf/1k+v8fHIw3NXL98XzpRBiE21lkP4VuNdjvOq6K65UbeSBrruAlBgSfoLDvrt3olj0sRBNWVhxRiPUjEbtQHh27nZhiAwQ3fiXDMW5U6N0G6cpPVC+meJ6ViYNVNHY5QRbp0GU/l69/uXj24tJcTvCiIURXkiRQSRziAiYvJ0bZ44A14UrtTd78rBKP3EocjSlIclU8p6D8TuLUJ87ai20CCXkWlJ7ZC4Rg+oZGqFXyWlOOsXhf5xlqWzrUHdF1oQoMRVWpH1FmQqnukzraXJP1uy3Rd2pI3W1BZWhI63mlg1XvnVFMqnEMPDnhAV1TcDlNrjlOT36150400YbbY7m7/ru6lDPF2lnvb3eDQw3hSOqRCjpL/S0tnVEBzIuVZhBqPr+vk1HKpQuUDlG0ThiKFzVaWWV4JqCf5aRYO0O3tPwZYhtbmeW896+lMnq9vUIZTtp38T0tZ2W/bndtlMIASHVdASIwsv95/uIlzS2ut08bYf0OFXToN0VhL2+ifHCJ3yLSUa6r5Uf2FLfZWD/SYFRrXAS0UtJPO/AvtJrTkGc6p6gUe20pe/KWBJ+QeucwsDpmKphiZgOOJttrUhhbHZquCuhEmnbmEmJ6y0L3Wf/QZDGWhdT4K43TVPv9SNzBAzceYwqt2Lj/PT7+ffADIN7vw991ffCfxvDY+N1AY3ZcZ15Hpbyfaph8prxO4p2Ab/hvpvSnuwIe96PYdbfI0iMorxkVwiENZoW5kup76JMMtiqBpX2ZJ4CMPELPkVfCXeN3/4kPeEP+GnFPPHsmnv5FtNEHjYsG0czxAZYRXtV+R0FtezJX3BwddDgsiiSwqhtJZkeKnOt/CVZvViO3fu6iNVodcWYd1P/v//4/FJOApuGNVoU6lDHQ5QUhT7dJQAieL65I/e/+7/47eRC8y86Ld1vjewflarVa+v/vvgYnjwYfrCIGyBrygVFhzwU0pizslvxmlOTUIDUs0eTP1DTtoP876P8O+d8P8l8uW8hHtwF7QP6zms1y/vdWt3mw//okf1IqAgEmdCJ0SY1vY7yaO8Mv0sIGWJYoQaP78kXsvt+Ks7dtvJq0bUrMW8jVQr5QBf9kQr0KRUNTgzGfRLMVxmM7x190d++ElIHAkS90rVZLZaoIM1dGKkbSzrIBmS8VC1Z2lFPOrNIC5IEsObv7wgDRVUEOZuQok7Vmoafgjhoqu0GuX5lsZzwPMDtqf6D5gU+Spnwh88JUZZKcaj4/zXD3uDhfghyZzN/DvcEKz91F2Nc4b/vcWWIIPJG6OFZFox9MpzRfXKbQpGXC1mNgx/MJhxxTqQwck4Rpx5TOlI5JqRwcM83gwOmDHJNTvBxOnQP/d+D//ln83yH+72fK/6FP/MpJ3D/CAeAB/u+k0SzH/21ZB/v/T6b/lypVUqj2RC4VKQcG0WZkRq0SEFRl8oEqJR64H94PTXHBqV4obJDMp5HF0jUrT9KgvB4G210AruH3qfASgXbmCyeMgaN4+QZt5N0Irx/043pNJMHCINXRLSYRJ8fFidApiCyUVQ8CytVcuXDJGUFcuOjp/e2dVe1Urca90B3/Ntepz5GN0YD3yfm3FuW8cXyVtWiB1vrj2/x9QcYZ8xWB5I9VKOBcwKI/iGUuR0A6E3s5aZVd6e9ho4lv+zAuOrsR+DBOusDRcmxoZv6qFK93dyew+HvY54bZ2c/d736DUQacyHX2M+SK9c37yqrbLo7RhY632AZGt6acn+JrYI29eE+XHF41P3NN9YEsOtlD9lTQbULKZoN7Kkdi0x6Qe7IO9jDmP+M1xbVI73RKebMcU06Uoy1cc1QNnCqGHR5zvtivkYOXUByeiUvKsc3YBg0Q8mD+JRCLci4J+5A81+OZjK1wd18UEBl8CrH5VwUDX6ucMGjCq0qleS+lj8m+/ZHPm3kmZGv5hxV3UW4X1smbYPoU8SWuFgZZ44B5HMcjTqB9uphEtTqnY9WzzE10tekHwt0kkUMVx4G0ibefFv0Z5tjIJSVGKqSWnudT/+WTQvWgiYKbwnwrevNcplRslZwUOMxCmhBJ2hVTWDP7vFeKCl1IwbMjmMYUk+dKI1oYEWIPxy5wb6lTqiuBbZ8DPu4EcmvX4j2Q0jVdw0Ia18rPv1bFz2+yqDUp4v8s4w4sMWUyu6nTDihYmyMOV8X1QB6DQ/KAz3vEn4mkHDv8518VZC9zwcHxxZsU5MUIy3zJidQRB8bScZwL8MMjvUAUsXo0kQ5/WA38vD8TT845JskZnlry2/mfitmWrr0qzzxLKflz7mKxNI881qnlKQUwoaRANt25UWj2S5TklajPEexwSVTYc4zBnkZf//nXwbU3LGVr32oeV3NXBPW9zcDsFYx/foOvOHh/EdiqMIbZwWNrQYFTKGZJOUzIdhh2dc/IRudprs5ZEbCAMTOMuU6MjYpvTmF90Idllu5Niu2R/aLYWAPJVHGmKpWS6hhgC8egKsIPv6ECqniNixTs2bN4HIs0PWs+mD2gJQ4LDxR9WbzaJUeWi564GFwN0ztRivICXBSQtQXmMcvfxyaUyJhCizcI/7ZrFWcOFYpznxXmTq9VwPkCMHB4BHEF59woKC9aXzTwnETkxHgchQz2sOTFxYKpGFvJw6Iet4QZYbkp+EZP6gKW/VhYWZew+9RUsWCdApLglaMaHmKsoRwMYk7fh7v56Ohuql18e3d1X4p45IecN32NtLplFFw1LpSjxj1GliPGdU9tGBfXx7y6QWjZBEzPt1++2VPj/E9phWVM9hW3aUlMK3FULA7bDctbHIJJsYIcwu5nGXGJL2B536aTNypbZCVP64tHPXLUdhSdCR9dHAfYeD4gELwyOQ+AyTnfdemngr2lke0xRyl5i9xpK9+Dodoq94QasI+RhTjakU25DTFh2mpZeI4ZW1W9FDCyEK47Z2yVrcm0rVraESfLVdnAsZqV+RnB5DC3bwFqxdFXd81IQhLY3V4+yytntYVeLs0JGiuoZK+UyprySZIVAgVFi1X2V6meRfsDTr7qAzt8UMIe9L+fVP97sP/+p+l/d8V/6R7iv3xu+t9ZuGJj1j9AAfyQ/Xer1SnFf2m1GtZB//uJ9L8vz39BJWi8ilxOfqbL4C/nwP4Bh/OmDfK9DCjHmdHlrXCwms1D5B5YIZIm9pIKLNJ5sKYsDl2QuyqUHO7XCyiNAVdyYcFRs7R0l0F0azsLikIN8okg41TUGLP1ZIaixPSIC9dZoP51hOFmUG+XV9uyspZ1tNLcO1XcUmjPT2beoBLIVwVHX+ZcXnEFo9OoQKO7gqMjf03R+mF9tAqHV88VnrmJzYGJbRocpn8jdYE2oxi7FKhYowqY34kq2vYaYIX6I/u+AkJYvMuueI95e9lyEgNGV3gqyKF7y4w7p5bxl42RVkmezs8cDZDRBALkxYsByBCSI0dGHblo/mlQJBpEGD3TLjHv/QDe6MiVN4+OWg3kxSu50miVmrBAJCsRQ40KHNRaqoXStwIMA+7/2rLPn7z5Hi0sps7YHQXBVZ1d+Wvs/F8LYZBomVtbrCdW9/TxEjlqDjSMHch5DXrtYWFQ8S2AGLpCs+ddqiVVsSpGcb/bxiJ7KlcAmjLYgIxHwLKPt5yREBOn0ksOzCjalrUlUrojQK2hmD0b4W0QrcV9BcNqu4VRuMswubXHznjOU/gAiFf+FjVfZQsoUKGs9SNkKgYJeG+SAFU6jSNOVeBRFlm8VIhRVOJgEWQcOgeKqlVu/G2LTFIuCocU6hibMVjanbKtPS3V+1c+yz4gl50l16LYurXUsPw5ZJBqgEzuLdaGUZaKN9+DJyzOpvJprF/kBEyjGFUolU3LnjFnOQH3Qoqs2u4DQTtIsQf7n4P89w/Y/4D812i0Qf476Z60Gwf7789J/kNSTH59SqX7EeXAh+y/u11Lyn+ddquD8l+n0Tzkf/tU8t/3sOpuVDJ66HGyZulLxq8oDbcXi/l04YUhXqRhONB47CzY/bbT+HfgUiOMwem7cSw2DfOEE3SHDvmVgSToVCJ4ECxrfoBJX5rN9gb+x6ziayfGm/NlsMZr9bHruyr1hnCI/2DHNRqKkQqHyh+UnxMGs3i4JQui6PdgerEqf7xGE2X69sKfO/7YrYx2OgOficifcQR1npYpL+NteE4iGdpelOwtptrdKHNk5YFLb1Zki1Z8mVv2H85VIffbnT7ExVtudckt52A+TZdGV7M0lx7l28ldhnNfd9eDR8RNUtS4ckx4AxhralOHJTZkd2YyB2HcBwFW1y2riWHumzijUXbbngWFB/iYaO01A4EUZe9mp1MVehP9TukfZEQ5OwvAdgUlT+Vl2mhmEjaxjyjWwAj9S/PGmyRz4GdB9uOG+PGcQsLycxwMXldmd/tZVioJ23rZzbmupd3GztpV0AnvNV4rdQl4d3R0DVVT07l7o5JxrnTFsnftM6TdZmHPRObMiJ7NH52n/e/B/zW3+b/Ggf/7JPxfN8//dU677Zb5uNvsnhy4v8+K/wuBEjqRa9Pp9pHvAB7g/7ptS+n/O1an28b4L9bJIf7Lp+L/nqLflnBEvHQWC3bGr6Eu03OLgWCIYcrFISA7bQw6gTxeGoogi0RgYA7bZ1kI9Fj8tVZDrRJ18dfU3T/m0OiYDQ2DLehwdE9DDuKeY7hY/Qwtvi7xqcSV6mSeO8LsOcdW59+B4cSwI1NmYuXokAHlumhxh7a4EzQZZvWa0KfAaCU4RHR1g+d/lXo3OP5VQ1lAm8gVP71+IzBiA0wOmV9kTpUhXMw/yZZGNo/2Vxgoh01E3QkB55K43HQmWGe6gjUoDg1LvnzTS43jgU2O3bGhbE4LQeY5lAKH1s/YZTZeJhMwv0baWlHPFUqnBbzQbbzb+D0LG8+3KRQ05+FsvX+HITotd96sORdU531encXyeS59p2F6iog7XSFb++3IMcjL7jqN3ZUkPu2sAxzyQ86jhfIYdnuXiXcp4I1j0vy3wt4A9uwMejPo8V0KNrMleZCJ0N7YRUVFblY8cxIoySkB8MnxdWadi/c8KWrr04WDJqbBVKWWJN5ZpaBMTZGlVd+0aNVHlTHQMeV5zeeg86ZQluwocRtgMz1q1CuHWM+VUvaSvTyfr8PIoAc0CZSerPxA90DesAz1mDJw5m2nc0GG6Z7Mozx0GEMLpR9S+wPsDTSnTTFTjp9CzpCV5XU8WHu9tUeWoWmx4ZlY4xUghRxivL+DMvV6vlCv0Zxw0CEZFQhHdkaDz+wlyf7VJSGSOs3g9zbcE4IoxbTtQETudT4UkeM/1EIunk2xidCfaTlrzzTxHeYtI2EyJ72GRi58ueMDbArXnGjziwIlmQPSoUF3NXgZcte4z/q4LkYzehtyZsHvq+I3FCCzQEZZ2moqzSF8tnYBT8UmrQZBpCqujZyZfYgyP2ODY0p6wVjxNkTg+fJuFEvmkmxep9JnKnhOtf1davAEjRDd6y2vrh6jEJYhY9weYgb8QtvbnkJxv4Di98P7ovFw4JEfBMC7aFmL9+6bmLUkdOikcEeDYKi1Mw/iJkank3aj0SM1kOev3EKxDWybjVUVt/B5a0HrG1Rm+HhvvpHJQ+FF+uxWPgOBelMVIfo1oFxtWh1xJPSNBVPbNAwSvdOnt/j0tlHK0/ghCw3H7h1M7F6t9UBmLbxt1MJbo4dj+h6HfQy/qiql4QZebvjlbzi3Y/g1LKUFJQIHqI/J93QJRy+FI6c9Fd8Af9bgbeBxCsYwGRazDXwA5pTmUECezDkof6W84evk7wF2v8G02vsTWD+IfXGKe+69yt5JnhI7T4oyJqazLiLidGtTe8Mt3JvmgpT9Wz5IWS+tL4OUTXeHSJPtAJFNT5qdYdjKJ2v9joBwr31AeDZ59OzrPg28Ns0l6cA9DHtQbd/KVji1nL7quppjJfL4XVBX7tBWZW3E+9uQNGn3nS3ru6BSeibe08mH4sBdMaYXNXv8KLVMeWRAWf5xxmWDa3hSVGxX+U2Mb+I8//15RgA72P8e7H8L+j+rg/GfT08O4R8+K/0f6Sr+kOyPHxD/66SR6f9auP+b7ZND/IdP8rcv/hfbzn42IQ2YCf7QgAZ5bROFMMhH6GKbuapy1dbIoA7K7AnHdbVTz9PZ48of7/a7LzvyUgxWMrxVrrQ5ERmq4Y/ypa6K0YWfV1L1kDJj0eARMVaPhve/J/iLGPVHQ8qRWMMHwK3Ln7HQydAbn7Irp3ph/J5QKvk+tYDf8E2Lsv+prqQ34B25HubtFWuJ0TMb03ux/GyDtf7L8n8H+79/Gv9XtP+z2ieW2bROu91G97DHPiP+L1r56EJh450fHBrzj73/35P/yWo1Vf7vzol1ckL3v932gf/7FH9f/lt95Pn1kRPPK1+K7yjmOt37Lp0k8jacSZvN7TnlgjhDR4WlE90KzBpO4TJUCUSiKTZhLoJZxd0QB/fm4slPl9+9vvjxxcWl/euLi6evL1+9+UvfRbuzirvWjTvhAvcktH5ffGWJr5riq5Z2JsLbZI6eY9vx6YTk7Ph22JtsbKiWRqUS5eu/VO30VZMn8FVL0J1hOuyv7qx7ePtVm983v/kPS7wTs8gNRe2F0P6T/R3eMT/3Dpk5AwZ4XxmI2jQbRMEbqI6+99zcULx7V55NMdqyUMGctscun/N4d3clZLha8aF+SYKuFwXCTPncYAykwrypRWIjfcqRWuiQ05HW2F5gZ0jcMzEJgIF01+IrX+Ss3M4Kj0QuMhfeQXU0YVOjUHWAxfqljofiP/5D3BXaUC2SKg8XZYI3bge6fuD/Dvzf38H/ddonp+bpSbv5+GAB+Nnxf+Tn+5F5vw/h/9pdxf91Tzj/Z6tldQ/836fn/ygVXC12E+YB9TQpm4Hu9sD3oWt6T8RXHjAQZNeDoQDYpA1NYpij4oTk5ocwgGfiaX+3v3mOu/rBRibj72Spvnq6g4niFlPO6X3R/XNcEsbeLwyLuZOPOzBu8+9h6vLMHLFuP1OW94z7SlkmYs5oIooFZoDYX/28fx57OfAUmIoDh5mlPLdsUAaClcaQRfa72Pf2yKRP+985thSe7x1dcTzFHpmlJBnlyQ8/PH/904t/oZPxwP8d+L8c/9dud7qm1e10Wu0D//c58X8cOSNcmXTsffz9/z7/D5X/CRWAjQ7lf+pYh/vfT/I38aZTOP9mXiIcQoI68R31iTuGzwgP1NHu5xU+ZpuPHWfSck2za40nk1FTWLTYlVqttq/FyvHx8d5Wv/1W1FqNRvVEHPMHPODI1G8ix4+Br1m60XMur/u++WMwWS3cUjBXiubqxsFiReGncsbcbN6dAIvkFmy8ty0Vp66T2GjoR9GO07bqdVl9u8Y4CKJJbM+r6tsNWW0vpqaN0Zr4oV6p7bKITHur5r9ygKc+RxGqHP/NFXWuieCwbb5Rt21dRoUydkWd4pSmGlACzdie4Y5HNL9xsAy5BZhlNOFYQDB5fRskH9AEx1RXsNd3zdCo7EPcMIg9XCgbRIkAI5VtofCuEhKZp63Tcbc5Ms3Tk67rtty9yLyzjS203lkKEbzTRPymf1P0PpeFX6gQa5ee7+7HcPzTQQxAuK0S1y6iaKtZFe95WbbMLVoHs8EtQt/z863go7i3jb+J68dBFKfxyf7LjYJYx6i4FiZl3YHKxjYu/y2N/GNovQ8DYdY3ICXpciQ7in0ppqsombsRLJmiJRMQ/jB6cZL6YxHu62/hYEG/sJErYmfq7kV7LD3A2Q0Vscg9Mqkf3TC5F30/2mfhn+wwCsZuDFPYwvw9hRTyn7Zbj6cd02yOx5NpZ7oX+fc1s4X/+wriFrDauAXoX/jJiYlx/dYeBqgzE0Xs04Bg62ZF7pRLaPZctSe3haZp+H9FMHpymgLP95I0TYHMP5dthT7M77SEmEin/ClQd3/s2mniO0o50mOc/dtb/sew9b0jKuISawz68vKm+C5HAfLHWakUjxjvXejLXmxbL4DOL0dAoLaRrPhO4ta4cTJ1H1umCULGZNR9vBe3SrW3UKr0njCpSdSUP1JyisHefv3hqTO+GgX76GjmlAX0DVAnzXAhSYGN0UdyBzYv99gJE85c6PlQj4N79zlXgQOUG186C6orn+45xv/B5j7G4b4DGhiyc01uWBRJz76J0FUk4pwedh4yth/YAF8qZuhFAqdm1U+nx2vVOqG1oo+/aa2ydjImrwyv7NVOwGULWdzz2Yp8nB27DVrGMJnvhVncZpdYXPpIAfErEb/X/uL2XxAIo1h6salZGLsAdNA6HPR/B/3fP0v/Z3Uty2x1H3e7pwf93+eo/0NGffSRs4A+5P9x0mql9n9tq4X2f81256D/+xR/IDOd44rLGO4krKFM644TdOn2xbPzX3oCg9eRgwFI4sAL1vgaTz7gRHioM6BgfYBAS8CofBgRDsNOniUUN4QjsO+OGEJPERtZnLHp+hTjpuSTxecES2mpVqxl7pI8U5eSvPi4HVsaw+bt7kVPOSbkcM4EZZ7LN6Yvc7w+lkHmSeMnGqX3y5c2lRgJX1CdCV/Ggb12yIsmZmaq1Lxyk9BkQOrq/nDUVTFdrOK5dGQZk/tM6lf8DDiwVeRux3pXvrT8qNHgYO9nyHhiqkEdW3j25NzGJG/2+etL+8fLF88wz1sDg58EV1V2kobSHGCkks+1l0X/w2bG6+RZsIBZTdnh+dnrH15f2E9fXjQvXj412IeFojgMNL631ijWdCTdhh0fVmahDZnB3I4Qjk/Occy8gjoMBB1lrHJBjrfIJUmoYfzV+aMfAt5i7KB+LAMv8AKE5BAv284vhJXBX+Mm9i+TpdJoaRzrAYYy0MgtJqYEYgOZawrjbGe56QqlMPmbF1MEl85wWFzzw+F2+Dv8Hf4Of4e/vX//D+iB0loAGAEA'
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE)), mode='r:gz').extractall('.')
sys.path.insert(0, W0); os.makedirs('results', exist_ok=True)
from vsearch.weights import discover
t=time.time(); Wt = discover('/kaggle/input', '/tmp/w'); Wt['assemble_s']=round(time.time()-t,1)
R['weights']={k:v for k,v in Wt.items() if k!='layout'}
ok = Wt.get('dinov3') and Wt.get('sam3_ckpt') and Wt.get('sam3_sha256_ok', False)
if not ok: R['errors'].append('weights missing / sha mismatch'); save(); raise SystemExit('stop')
os.environ['DINOV3_PATH']=Wt['dinov3']; os.environ['SAM3_CKPT']=Wt.get('sam3_ckpt') or ''; os.environ['SAM3_STRICT']='1'
run('nvidia_smi','nvidia-smi --query-gpu=name,memory.total --format=csv'); print(R['weights'])

In [ ]:
# EG2 weights: flat 'eg2files__a@b' -> /tmp/eg2/a/b
EG2=None
for p in glob.glob('/kaggle/input/**/eg2files__*', recursive=True):
    rel=os.path.basename(p).split('__',1)[1].replace('@','/'); d=os.path.join('/tmp/eg2',os.path.dirname(rel)); os.makedirs(d,exist_ok=True)
    if not os.path.exists('/tmp/eg2/'+rel): os.symlink(p,'/tmp/eg2/'+rel)
    EG2='/tmp/eg2'
R['eg2_path']=EG2
run('install','pip -q install faiss-cpu pycocotools psutil && git clone -q https://github.com/facebookresearch/sam3.git /tmp/sam3 && pip -q install -e /tmp/sam3 && cd /tmp/sam3 && git rev-parse HEAD', 900)
run('install_eg2','pip -q install --target /tmp/eglib "transformers>=5.20" "sentence-transformers>=6.2" && python -c "import sys;sys.path.insert(0,\'/tmp/eglib\');import transformers,sentence_transformers as s;print(transformers.__version__,s.__version__)"', 900)
os.environ['SAM3_BPE']='/tmp/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz'
print({k:(v['rc'],v['tail'][-200:]) for k,v in R['stages'].items() if k.startswith('install')})

In [ ]:
run('data','wget -q https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip -O /tmp/davis.zip && mkdir -p data && unzip -q /tmp/davis.zip -d data && rm /tmp/davis.zip && python scripts/prepare_davis.py && python scripts/make_hard_queries.py && python scripts/add_distractors.py', 1200)
print(R['stages']['data']['rc'])

In [ ]:
T0=time.time(); BUDGET=100*60
def left(): return BUDGET-(time.time()-T0)
SAN=["import json, glob, sys, os, time, torch; sys.path.insert(0, '.')",
 "from PIL import Image",
 "from vsearch.core import sample_frames",
 "from vsearch.sam3_regions import SAM3Regions",
 "sam = SAM3Regions(); out = {'load_info': sam.load_info, 'precision': 'fp32', 'prompts': list(sam.prompts), 'threshold': sam.th}",
 "im = Image.open('/tmp/sam3/assets/images/truck.jpg').convert('RGB')",
 "out['official_truck'] = sam.raw(im, 'truck'); out['official_object'] = sam.raw(im, 'object')",
 "fr = [im for v in sorted(glob.glob('data/davis_bench_large/videos/*.mp4'))[:5] for _, im in list(sample_frames(v, 1.0))[:1]]",
 "t = time.time(); n = [len(sam(f)) for f in fr]; out['davis5_boxes'] = n; out['davis5_s_per_frame'] = round((time.time()-t)/len(fr), 3)",
 "out['ok'] = out['official_truck']['n_ge_th'] > 0 and sum(n) > 0",
 "json.dump(out, open('results/sanity.json', 'w'), indent=1); print(out)"]
open('sanity.py','w').write(chr(10).join(SAN))
run('sam_sanity','python sanity.py', 900)
try: R['sam_sanity']=json.load(open('results/sanity.json'))
except Exception as e: R['sam_sanity']={'ok':False,'error':repr(e)}
SAM_OK=bool(R['sam_sanity'].get('ok')); save(); print(R['sam_sanity'])
if not SAM_OK: R['errors'].append('SAM sanity failed -> SAM stages skipped (AMG fallback not available for SAM 3 image model in this repo)')
run('gpu_bench','python scripts/gpu_bench.py', 900)
try: R['gpu_bench']=json.load(open('results/gpu_bench.json'))
except Exception as e: R['errors'].append('gpu_bench json: %r'%e)
save(); print(R.get('gpu_bench'))

In [ ]:
R['index']={}; R['eval']={}; R['skipped']=[]
SETS={'large70':'data/davis_bench_large','small10':'data/davis_bench'}; D=os.environ['DINOV3_PATH']
EGENV='PYTHONPATH=/tmp/eglib TRANSFORMERS_VERBOSITY=error'
for sn,B in SETS.items():
    reg='grid+sam3' if SAM_OK else 'grid'
    run(f'index_dinov3_{sn}',f'python scripts/build_index.py --videos {B}/videos --out data/{sn}_d3 --model {D} --regions {reg}', 2400)
    if EG2: run(f'index_eg2_{sn}',f'{EGENV} python scripts/build_index.py --videos {B}/videos --out data/{sn}_eg2 --model {EG2} --regions grid', 1500)
for d in glob.glob('data/*_d3')+glob.glob('data/*_eg2'):
    try: m=json.load(open(d+'/meta.json')); m.pop('meta',None); R['index'][os.path.basename(d)]=m
    except Exception as e: R['errors'].append(f'index meta {d}: {e!r}')
save(); print(R['index'])

In [ ]:
for sn,B in SETS.items():
  for Q in ['object_hard','object']:
    jobs=[('dinov3', f'--index data/{sn}_d3 --modes '+('frame,region,both,sam,frame+sam,fuse' if SAM_OK else 'frame,region,both'), '')]
    if SAM_OK: jobs.append(('dinov3_rerank_sam_top20', f'--index data/{sn}_d3 --modes both --rerank samidx --videos {B}/videos', ''))
    if EG2: jobs.append(('eg2', f'--index data/{sn}_eg2 --modes frame,region,both', EGENV))
    for tag,cmd,env in jobs:
        if not os.path.exists(cmd.split()[1]+'/meta.json'): R['skipped'].append(f'{sn}/{tag}/{Q} (no index)'); continue
        if left()<120: R['skipped'].append(f'{sn}/{tag}/{Q} (time)'); continue
        o=f'results/E_{sn}_{tag}_{Q}.json'; run(f'eval_{sn}_{tag}_{Q}', f'{env} python scripts/evaluate.py {cmd} --queries {B}/queries_{Q}.json --out {o}', max(120,int(left())))
        try: R['eval'][f'{sn}/{tag}/{Q}']=json.load(open(o))['results']
        except Exception as e: R['errors'].append(f'eval {sn}/{tag}/{Q}: {e!r}')
        save()
R['finished']=time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()); save()
print(json.dumps(R['eval'], indent=1)[:4000]); print('errors', R['errors'], 'skipped', R['skipped'])
for f in os.listdir('/kaggle/working'):
    if f!='de12_results.json': subprocess.run(['rm','-rf','/kaggle/working/'+f])
print(os.listdir('/kaggle/working'))